# FingerBall Stage 2I-2 — MANO 3D hand mesh fitting (single camera, release focus)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2I2_MANO_3D_Mesh_Fitting.ipynb)

**目的**：Stage 2Hで目視確認済みの手指21点（185–198フレーム）を**一切上書きせず**、左手MANOの形状・姿勢を時系列で推定する。

- 入力：`hand_21_landmarks_2d_corrected.csv`＋必要に応じて元MP4。
- 観測重み：手動修正1.0、目視確認済み引継ぎ候補0.65、AI点0.35、遮蔽・欠損0。
- MANO 16関節＋5つの指先頂点をMediaPipe順21点へ並べ替え。コードで関節対応を検証。
- 目的関数：**2D再投影損失（Huber）＋PCA姿勢事前分布＋形状事前分布＋時間連続性**。単眼で奥行き・スケールは一意に決まらないため、**弱透視（weak-perspective）投影のモデル依存3D**。
- 出力：メッシュHTMLビューア、2D重畳品質管理PNG、再投影誤差CSV、21点3D CSV、関節2D/3D比較CSV、フィットパラメータ（.npz）、設定JSON。**モデル本体やMANOの.pklは出力・再配布しません**。

### 重要：MANOモデルのライセンス
MANOモデル本体は登録と許諾が必要です。[MANO公式サイト](https://mano.is.tue.mpg.de/)でご自身の研究目的に適合するライセンスを確認し、`MANO_LEFT.pkl`を取得してください。公式モデルファイルはGitHubや共有ZIPへアップロードしないでください。公開物にメッシュ等の派生データを含める場合も、許諾条件をご確認ください。

**本Notebookは探索的モデルフィットです。推定された関節角度や奥行きは実測値ではありません。**

## 0. Python環境・Google Drive

In [ ]:
%pip -q install "smplx>=0.1.28" "plotly>=5.18" "pandas>=2.0" "scipy>=1.10" "matplotlib>=3.7" "opencv-python-headless>=4.8"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display
import os,json,hashlib,math
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
import torch,smplx,cv2
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
print('Torch',torch.__version__,'smplx',getattr(smplx,'__version__','installed'))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)

## 1. 入力設定と凍結データの読み込み

MANOモデルを `/content/drive/MyDrive/母指回内トラッキング/mano_models/MANO_LEFT.pkl` に配置してください。公式ダウンロードしたモデルを複製・再配布しません。`INPUT_CSV`はStage 2Hの確定CSVです。今回手動確認された`propagated_suggestion`は推定候補であるという由来を残して利用します。

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
INPUT_CSV=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'hand_21_landmarks_2d_corrected.csv'
MANO_MODEL_ROOT=PROJECT/'mano_models'  # MANO_LEFT.pkl in this directory
ORIGINAL_VIDEO=PROJECT/(VIDEO_STEM+'.mp4') # optional QC overlays
OUT_DIR=PROJECT/'FingerBall_v2_stage2i2_results'/VIDEO_STEM
F_START,F_END=185,198
RELEASE_REFERENCE_FRAME=190
HAND='left'
PCA_COMPONENTS=12
STEPS_INITIAL=300
STEPS_REFINE=500
LEARNING_RATE=0.035
RANDOM_SEED=20261031
STATUS_WEIGHTS={'manual_visible':1.0,'propagated_suggestion':0.65,'ai':0.35,
                'interpolated':0.25,'occluded':0.0,'missing':0.0,'outside_edit_window':0.0}
if HAND!='left':raise ValueError('この投球データは左手として確定されています')
if not INPUT_CSV.is_file():raise FileNotFoundError('Stage 2H確定CSVが見つかりません：'+str(INPUT_CSV))
model_file=Path(MANO_MODEL_ROOT)/'MANO_LEFT.pkl'
if not model_file.is_file():
    raise FileNotFoundError(
      'MANO_LEFT.pkl がありません。 https://mano.is.tue.mpg.de/ で登録・許諾確認して取得し、'
      +str(model_file)+' に置いてください。モデル本体はGitHubに置かないでください。')
OUT_DIR.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(INPUT_CSV)
required={'frame','landmark_id','x_px','y_px','annotation_status'}
if missing:=required-set(df.columns):raise ValueError('CSVの不足列: '+str(missing))
if df.duplicated(['frame','landmark_id']).any():raise ValueError('CSVに重複キーがあります')
df=df[df.frame.between(F_START,F_END)].copy()
frames=np.arange(F_START,F_END+1)
if not all(df.groupby('frame').landmark_id.nunique().reindex(frames,fill_value=0).eq(21)):
    raise ValueError('185–198フレームの各21点が必要です')
df['frame']=df.frame.astype(int);df['landmark_id']=df.landmark_id.astype(int)
W,H=1206,648
if 'x_norm' in df and 'y_norm' in df:
    for px,normalized,extent in [('x_px','x_norm',W),('y_px','y_norm',H)]:
        part=df[[px,normalized]].dropna()
        part=part[part[normalized].abs()>.001]
        if len(part)>10 and abs(float(np.median(part[px]/part[normalized]))-extent)>3:
            raise ValueError('入力CSVと1206x648の画像座標が一致しません')
original_hash=hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()
print('Frozen CSV SHA256:',original_hash)
raw_xy=np.full((len(frames),21,2),np.nan,np.float32)
weights=np.zeros((len(frames),21),np.float32)
status_matrix=np.empty((len(frames),21),dtype=object)
for ti,fr in enumerate(frames):
    block=df[df.frame==fr].set_index('landmark_id').reindex(range(21))
    coords=block[['x_px','y_px']].to_numpy(np.float32)
    stat=block.annotation_status.fillna('missing').to_list()
    if any(x not in STATUS_WEIGHTS for x in stat):
        raise ValueError('Unknown annotation status: '+str(set(stat)-set(STATUS_WEIGHTS)))
    raw_xy[ti]=coords
    weights[ti]=[STATUS_WEIGHTS[x] if np.isfinite(coords[j]).all() else 0.
                 for j,x in enumerate(stat)]
    status_matrix[ti]=stat
obs_normalized=np.nan_to_num((raw_xy-np.array([W/2,H/2],np.float32))/W,nan=0.0)
obs=torch.tensor(obs_normalized,device=DEVICE)
w=torch.tensor(weights,device=DEVICE)
if float(w.sum())<1:raise ValueError('有効な観測ランドマークがありません')
print('Frames:',len(frames),'Weighted points:',round(float(w.sum()),2),
      'Statuses:',pd.Series(status_matrix.ravel()).value_counts().to_dict())

## 2. MANOモデル構築・MediaPipe21点への対応

`smplx`のMANO関節は16点です。指先は公式実装の頂点ID（thumb 744, index 320, middle 443, ring 554, pinky 671）から追加します。**MANO関節配列→MediaPipe配列への変換**を必ず経由します。

In [ ]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
T=len(frames)
try:
    mano=smplx.create(str(MANO_MODEL_ROOT),model_type='mano',is_rhand=False,
         use_pca=True,num_pca_comps=PCA_COMPONENTS,flat_hand_mean=True,
         num_betas=10,create_global_orient=False,create_hand_pose=False,
         create_betas=False,create_transl=False,batch_size=T).to(DEVICE)
except Exception as err:
    raise RuntimeError('MANOモデル読み込みに失敗しました。公式MANO_LEFT.pklか、smplx互換性を確認してください') from err
for param in mano.parameters():param.requires_grad_(False)
# smplx MANO 16 joints: wrist, index 3, middle 3, pinky 3, ring 3, thumb 3.
# Append vertices in order thumb/index/middle/ring/pinky, then reorder into MediaPipe 21.
TIP_VERTEX_IDS=[744,320,443,554,671]
MANO_TO_MEDIAPIPE=[0,13,14,15,16,1,2,3,17,4,5,6,18,10,11,12,19,7,8,9,20]
assert len(MANO_TO_MEDIAPIPE)==21 and set(MANO_TO_MEDIAPIPE)==set(range(21))
FACES=np.asarray(mano.faces,dtype=np.int32)
def run_mano(global_orient,hand_pose,betas):
    result=mano(global_orient=global_orient,hand_pose=hand_pose,
                betas=betas.expand(T,-1),return_verts=True)
    vert=result.vertices
    joints=torch.cat([result.joints,vert[:,TIP_VERTEX_IDS]],dim=1)
    if joints.shape[1]!=21:raise RuntimeError(f'MANO joints unexpected: {joints.shape}')
    return vert,joints[:,MANO_TO_MEDIAPIPE,:]
with torch.no_grad():
    v0,j0=run_mano(torch.zeros((T,3),device=DEVICE),
                   torch.zeros((T,PCA_COMPONENTS),device=DEVICE),
                   torch.zeros((1,10),device=DEVICE))
print('MANO mesh:',tuple(v0.shape),'21 joints:',tuple(j0.shape),'faces:',FACES.shape)
print('MANO left model ready')